# 81장 · 실행 추적과 기준선 설정

7월 1일 문의에 지난달 기한인 30일을 안내했습니다. 검색기가 구판을 골랐는지, 모델이 개정판을 읽고도 잘못 답했는지 실행을 나눠 살펴봅니다.

LangSmith에서 검색·모델·도구 실행을 따라가고 실패가 처음 발생한 단계를 찾는다. 데이터셋·모델·프롬프트·색인 버전과 기준선 실험을 기록한다.

## 어느 단계에서 구판이 들어왔나요?

In [ ]:
editions = [("A1", "2026-06-01", 30), ("A2", "2026-07-01", 90)]
def retrieve(as_of):
    eligible = [row for row in editions if row[1] <= as_of]
    return eligible[0] if eligible else None

def answer(as_of):
    doc = retrieve(as_of)
    return {"citation": doc[0], "days": doc[2]} if doc else {"citation": None}

print(retrieve("2026-07-01"))
print(answer("2026-07-01"))

('A1', '2026-06-01', 30)
{'citation': 'A1', 'days': 30}


두 번째 문서도 7월 1일부터 유효합니다. 첫 번째 후보를 그대로 돌려주면서 A1이 선택됐고 답변은 그 값을 그대로 썼습니다. 검색 결과에서 이미 오류를 볼 수 있습니다.

## 검색 단계에서 처음 틀린 값을 기록하기

투두 앱에서 409를 의도적으로 만든 요청과 잘못된 답을 낸 검색 요청을 구별해 기록합니다. 상품설명서가 7월에 개정됐는데도 30일을 안내한 원인을 찾으려면 최종 답 앞의 검색 결과부터 열어야 합니다.

노트북을 같은 수업 폴더에 모아 열거나 압축을 푼 장별 폴더에서 실행하세요. 시작 셀은 service-lab/approval.json을 먼저 엽니다. 파일이 없을 때만 셀에 적힌 복귀 예시를 읽습니다.

In [ ]:
from pathlib import Path
import json
here = Path.cwd()
# 각 장 폴더에서 열거나 노트북을 한 폴더에 모아 두어도 같은 실습 폴더를 씁니다.
course = here.parent if here.name in ("daily",) or (here.name.startswith("chapter") and here.name[7:].isdigit()) else here
lab = course / "service-lab"
lab.mkdir(exist_ok=True)
incoming = course / "service-lab/approval.json"
if incoming.exists():
    previous = json.loads(incoming.read_text(encoding="utf-8"))
    print("내 실습 파일", incoming.name)
else:
    previous = {'item_id': 2,
     'statuses': [200, 409],
     'version': 1,
     'done': True,
     'mode': 'local-version-test',
     'events': [('read', 2), ('approve', 2), ('write', 2)],
     'writes': [2]}
    print("복귀 예시", incoming.name)

내 실습 파일 approval.json


## 검색 단계에서 처음 틀린 값을 기록하기 실습

이번에는 앞 장에서 남긴 실행 기록에 같은 검사를 적용합니다.

In [ ]:
editions = [("A1", "2026-06-01", 30), ("A2", "2026-07-01", 90)]
as_of = "2026-07-01"
eligible = [row for row in editions if row[1] <= as_of]
chosen = eligible[0]
result = {"case_id": "july-deadline", "as_of": as_of, "ranked": [row[0] for row in eligible],
          "observed": chosen[2], "expected": 90, "gold": "A2", "mode": "local",
          "editions": editions, "approval": previous, "release_version": "baseline-v1"}
print(chosen[0], result["observed"], previous["statuses"])

A1 30 [200, 409]


eligible의 첫 항목을 쓴 줄에서 구판 A1이 골라졌습니다. 검색 결과는 틀렸지만 그 문서를 읽어 돌려준 30일은 A1 원문과 같습니다. 마지막에 표시한 두 상태 코드는 앞 장의 승인 시험 결과입니다. 이 로컬 기록을 아래 수업용 LangSmith 추적에서 본 부모·자식 실행과 대조하세요.

### 직접 해 보기

기준일을 6월 30일로 바꾸고 유효한 판본 중 시행일이 가장 늦은 항목을 고르세요. 6월에는 A2를 쓰면 안 됩니다.

In [ ]:
# date가 기준일 이하인 항목을 추린 뒤 시행일을 기준으로 max를 적용하세요.

<details class="notebook-answer"><summary>정답 코드와 실행 결과</summary><pre><code>possible = [row for row in editions if row[1] &lt;= &quot;2026-06-30&quot;]
print(max(possible, key=lambda row: row[1])[0])</code></pre><pre>A1</pre><p>7월 1일에 시행한 A2는 6월 30일의 유효 후보에 들어가지 않습니다.</p></details>

### 82장에서 다시 열 파일

방금 계산한 값을 service-lab/evaluation/baseline.json에 저장합니다. 같은 장에서 조건을 바꾸고 다시 저장하면 다음 장도 그 값을 읽습니다.

In [ ]:
destination = lab / "evaluation/baseline.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(destination.name)

baseline.json


### 직접 해 보기

기준일에 유효한 문서 중 가장 최근 문서를 고르세요. 6월 30일, 7월 1일, 최초 시행 전날에 각각 무엇을 반환하는지 확인하세요.

In [ ]:
# retrieve를 다시 정의하고 세 날짜를 검사하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre style="white-space:pre-wrap;overflow:auto"><code>def retrieve(as_of):
    eligible = [row for row in editions if row[1] &lt;= as_of]
    return max(eligible, key=lambda row: row[1], default=None)

for day in [&quot;2026-06-30&quot;, &quot;2026-07-01&quot;, &quot;2026-05-31&quot;]:
    print(day, answer(day))</code></pre><pre style="white-space:pre-wrap;overflow:auto"><code>2026-06-30 {&#x27;citation&#x27;: &#x27;A1&#x27;, &#x27;days&#x27;: 30}
2026-07-01 {&#x27;citation&#x27;: &#x27;A2&#x27;, &#x27;days&#x27;: 90}
2026-05-31 {&#x27;citation&#x27;: None}
</code></pre></details>

## 수업 서비스에 적용하기

수업용 LangSmith 프로젝트와 인증 정보를 설정하고 langsmith 패키지를 준비하세요. 아래 셀은 현재 커널의 두 함수를 한 요청의 부모·자식 실행으로 기록합니다. 추적을 켠 뒤 기준선과 수정본을 각각 실행해 입력, 검색 결과, 최종 응답을 비교하세요. 팀의 그래프에서는 검색·모델·도구 노드를 열고 데이터셋·모델·프롬프트·색인 버전과 실행 링크를 함께 남기세요.

In [ ]:
run_integration = False
if run_integration:
    from langsmith import traceable, tracing_context
    
    traced_search = traceable(name="상품설명서 검색", run_type="retriever")(retrieve)
    @traceable(name="문의 처리")
    def traced_answer(as_of):
        doc = traced_search(as_of)
        return {"citation": doc[0], "days": doc[2]} if doc else {"citation": None}
    
    with tracing_context(enabled=True, project_name="course-product-qa"):
        print(traced_answer("2026-07-01"))

### 제출 전 확인

LangSmith에서 검색·모델·도구 실행을 따라가고 실패가 처음 발생한 단계를 찾는다. 데이터셋·모델·프롬프트·색인 버전과 기준선 실험을 기록한다.

[공식 문서](https://docs.langchain.com/langsmith/annotate-code)